In [73]:
import services.utils as ut
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from nltk.stem import WordNetLemmatizer
from gensim.parsing.preprocessing import STOPWORDS
from gensim.utils import simple_preprocess
import random
import services.utils as ut
import services.model as md
import joblib
import json
np.random.seed(42)
random.seed(42)

In [74]:
def predictionoflabels(predictions, jsonfile):
    with open(jsonfile, 'r') as f:
        labels_dict = json.load(f)
    for pred in predictions:
        label_name = labels_dict.get(str(pred), "Unknown Cluster")
        print(f"Cluster {pred}: {label_name}")

In [75]:
def printlabels(jsonfile):
    with open(jsonfile, 'r') as f:
        labels_dict = json.load(f)
    for key, value in labels_dict.items():
        print(f"Cluster {key}: {value}")

In [76]:
custom_words = {
    'please', 'help', 'assist', 'support', 'thanks', 'thank','soon','mentioned',
    'im', 'ive', 'us','would', 'could', 'need', 'want', 'trying',
    'tried','check', 'checked', 'make', 'made', 'get', 'getting','also',
    'use', 'using', 'used','thing', 'something', 'anything', 'everything',
    'way', 'time','issue', 'problem', 'request', 'work', 'working', 'fine',
    'available', 'recent', 'recently','facing', 'doe', 'noticed', 'happening',
    'started', 'happen','different', 'steps', 'did', 'regards','already', 'multiple',
    'last','times','followed', 'reviewed','specific', 'possible', 'related','new',
    'old','find', 'try', 'say', 'mean','name', 'email', 'price', 'one', 'unresolved',
    'add','note', 'may', 'dont', 'know','sure', 'changes', 'performed', 'properly',
    'original','like', 'similar','reported','doesnt', 'sometimes', 'acts', 'works',
    'ensure', 'desired', 'action', 'remains', 'life', 'seems', 'might', 'guide',
    'much', 'others', 'heavily', 'daily', 'task', 'affecting', 'assistance','hoping',
    'persists','didnt','option', 'perform', 'recommendation', 'information', 'official',
    'solution', 'provide', 'making', 'user', 'customer', 'item', 'device','far', 'luck',
    'contact', 'contacted', 'occurring','resolve', 'function', 'came', 'having', 'change',
    'haven', 'let', 'unable', 'able', 'afterward', 'var', 'step', 'order'
}



In [77]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\{.*?\}|\[.*?\]|<.*?>|\(.*?\)', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text) # remove URLs
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' ', text) # remove email addresses
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # remove non ASCII characters
    text = re.sub(r'[^a-z\s]', '', text) 
    text = re.sub(r'\b[a-zA-Z]{1,2}\b', '', text)  # remove very short words
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [78]:
lemmatizer = WordNetLemmatizer()
final_stopwords = STOPWORDS.union(custom_words)
custom_words_lemma = set([lemmatizer.lemmatize(w.lower()) for w in final_stopwords])

def preprocess(text):
    text = str(text).lower()
    tokens = simple_preprocess(text, deacc=True)
    processed_tokens = []
    for word in tokens:
        lemma = lemmatizer.lemmatize(word)
        if (lemma not in custom_words_lemma and len(lemma) > 2 and lemma.isalnum()):
            processed_tokens.append(lemma)
    
    return " ".join(processed_tokens)

In [79]:
def nlp_cleaner(text_list):
    results = []
    for text in text_list:
        cleaned = clean_text(text) 
        processed = preprocess(cleaned)
        results.append(processed)
    return results

In [80]:
def clean_for_embeddings(text):
    text = text.lower()
    text = re.sub(r'\{.*?\}|\[.*?\]|<.*?>|\(.*?\)', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text) # remove URLs
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' ', text) # remove email addresses
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # remove non-ASCII characters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [81]:
def nlp_cleaner_embeddings(text_list):
    results = []
    for text in text_list:
        cleaned = clean_for_embeddings(text) 
        results.append(cleaned)
    return results

In [82]:
raw_data = ["I need help with my account", "Software is crashing in the production environment"]

## Kmeans

### Kmean TFIDF without SVD

In [83]:
Kmeans_tfidf_without_svd_prediction_file = ut.loadingfile('kmeans', 'kmeanstfidfwithoutsvd_cluster_labels.json')
printlabels(Kmeans_tfidf_without_svd_prediction_file)

Cluster 0: Firmware Stability and Freezes
Cluster 1: General Productivity and Power Config
Cluster 2: Accidental File Deletion and Recovery
Cluster 3: Software Version and Crash Reports
Cluster 4: Unexpected UI Error Messages
Cluster 5: Account Security and Password Recovery
Cluster 6: Security Privacy and Screen Flickering
Cluster 7: Intermittent Performance and Returns
Cluster 8: App Cache and Maintenance
Cluster 9: Community Forum and Online Tutorials
Cluster 10: Hardware Noise and Fault Suspicions
Cluster 11: Website Configuration
Cluster 12: Productivity Impact and Reliability
Cluster 13: Factory Reset and Connection Failures
Cluster 14: Network and Home WiFi Stability
Cluster 15: Battery Degradation
Cluster 16: Cables and Peripheral Accessories


In [84]:
Kmeans_tfidf_without_svd = ut.load_model('kmeans', 'kmeans_tfidf_without_pca.joblib')
predictions = Kmeans_tfidf_without_svd.predict(raw_data)
predictionoflabels(predictions, Kmeans_tfidf_without_svd_prediction_file)

Cluster 5: Account Security and Password Recovery
Cluster 3: Software Version and Crash Reports


Prediction  
Cluster 5: Account Security and Password Recovery  
Cluster 3: Software Version and Crash Reports

### Kmean TFIDF with SVD

In [85]:
kmeans_tfidf_svd_prediction_file = ut.loadingfile('kmeans', 'kmeanstfidfsvd_cluster_labels.json')
printlabels(kmeans_tfidf_svd_prediction_file)

Cluster 0: Account Display and Access
Cluster 1: General Account Support
Cluster 2: Billing and Android Support
Cluster 3: Admin and Account Locking
Cluster 4: Order History and Shipping
Cluster 5: Factory Resets
Cluster 6: Online Profile Authorization
Cluster 7: Software Productivity Profiles
Cluster 8: Address and Location Management
Cluster 9: Security Credentials and Lockouts
Cluster 10: Application Stability and Battery
Cluster 11: Account Recovery History
Cluster 12: Troubleshooting Feedback
Cluster 13: General Account Inquiries
Cluster 14: System Setup and Initialization
Cluster 15: Peripheral and Adapter Connectivity
Cluster 16: Software Permissions and Settings
Cluster 17: Third-Party Integration
Cluster 18: Cloud Services and Billing Address


In [86]:
kmeans_tfidf_svd = ut.load_model('kmeans', 'kmeans_tfidf_with_svd.joblib')
predictions = kmeans_tfidf_svd.predict(raw_data)
predictionoflabels(predictions, kmeans_tfidf_svd_prediction_file)

Cluster 8: Address and Location Management
Cluster 11: Account Recovery History


Prediction  
Cluster 8: Address and Location Management  
Cluster 11: Online Profile Authorization

### Kmean Embeddings without PCA

In [87]:
kmeans_embeddings_without_pca_prediction_file = ut.loadingfile('kmeans', 'kmeansembeddingswithoutpca_cluster_labels.json')
printlabels(kmeans_embeddings_without_pca_prediction_file)

Cluster 0: Intermittent Performance and Battery Drain
Cluster 1: Data Privacy and Security Concerns
Cluster 2: Data Recovery and Hardware Replacement
Cluster 3: Account Lockout and Password Reset Failures
Cluster 4: Software Bugs and Application Crashes
Cluster 5: Firmware Updates and Consistently Occurring Issues
Cluster 6: Power Failures and Support Responsiveness
Cluster 7: Credential Errors and Widespread Device Issues
Cluster 8: UI Error Messages and Screen Pop-ups
Cluster 9: WiFi Connectivity and Network Detection
Cluster 10: Screen Flickering and Hardware Troubleshooting
Cluster 11: Charging Failures and Original Accessories
Cluster 12: Network Setup and App Persistence Issues
Cluster 13: Manual Configuration and Factory Resets
Cluster 14: Version Updates and Intermittent Glitches
Cluster 15: Strange Noises and Suspected Hardware Defects
Cluster 16: Purchasing Assistance and Navigation Help
Cluster 17: Software Freezing and Security Stability
Cluster 18: Stable Internet and Disc

In [88]:
kmeans_embeddings_without_pca = ut.load_model('kmeans', 'kmeans_embeddings_without_pca.joblib')
predictions = kmeans_embeddings_without_pca.predict(raw_data)
predictionoflabels(predictions, kmeans_embeddings_without_pca_prediction_file)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cluster 7: Credential Errors and Widespread Device Issues
Cluster 4: Software Bugs and Application Crashes


Prediction  
Cluster 7: Credential Errors and Widespread Device Issues  
Cluster 4: Software Bugs and Application Crashes

### Kmean Embeddings with PCA

In [89]:
kmeans_embeddings_with_pca_prediction_file = ut.loadingfile('kmeans', 'kmeansembeddingswithpca_cluster_labels.json')
printlabels(kmeans_embeddings_with_pca_prediction_file)

Cluster 0: Account Recovery and Invalid Credentials
Cluster 1: System Freezes and Error Pop-ups
Cluster 2: Power-On Failures and Boot Issues
Cluster 3: Factory Reset and Persistent Product Issues
Cluster 4: Navigation Support and Action Guidance
Cluster 5: Intermittent Performance and Battery Drain
Cluster 6: Configuration and Manual Troubleshooting
Cluster 7: Data Recovery and Syncing Conflicts
Cluster 8: Home Wi-Fi Network Detection
Cluster 9: Application Crashes and Software Bugs
Cluster 10: Data Security and Privacy Concerns
Cluster 11: Charging Failures and Power Supply
Cluster 12: Screen Flickering and Display Hardware
Cluster 13: Locked Accounts and Browser Authentication
Cluster 14: Firmware Update Post-Install Issues
Cluster 15: Initial Network Setup Failures
Cluster 16: Intermittent Internet Disconnections
Cluster 17: Mechanical Noises and Hardware Defects
Cluster 18: Support Escalation and Reset Failures


In [90]:
kmeans_embeddings_with_pca = ut.load_model('kmeans', 'kmeans_embeddings_with_pca.joblib')
predictions = kmeans_embeddings_with_pca.predict(raw_data)
predictionoflabels(predictions, kmeans_embeddings_with_pca_prediction_file)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cluster 0: Account Recovery and Invalid Credentials
Cluster 9: Application Crashes and Software Bugs


Prediction  
Cluster 0: Account Recovery and Invalid Credentials  
Cluster 9: Application Crashes and Software Bugs

## HAC

### HAC without TFIDF

In [91]:
hac_tfidf_without_svd_prediction_file = ut.loadingfile('hac', 'HACtfidfwithoutsvd_cluster_labels.json')
printlabels(hac_tfidf_without_svd_prediction_file)

Cluster 0: General Maintenance and Product Inquiries
Cluster 1: Firmware and Software Version Updates
Cluster 2: Account Recovery and Password Reset
Cluster 3: Mechanical Noise and Hardware Suspicions
Cluster 4: Screen Pop-ups and UI Error Messages
Cluster 5: Network Connection Failures
Cluster 6: Intermittent Internet and Router Stability
Cluster 7: File Retrieval and Document Loss
Cluster 8: Critical Power-On and Boot Failures
Cluster 9: Application Crashes and Version Errors
Cluster 10: Accidental Deletion and Data Recovery
Cluster 11: Battery Degradation and Life Cycle
Cluster 12: System Crash and Data Loss Recovery
Cluster 13: Invalid Login Credentials and Access
Cluster 14: Software Freezing and Glitch Stability
Cluster 15: Home Wi-Fi Detection and Setup
Cluster 16: Hardware Repair and Replacement Requests
Cluster 17: Installation Errors and Unexpected Loss
Cluster 18: Data Security and Privacy Concerns


### HAC with TFIDF

In [92]:
hac_tfidf_without_svd_prediction_file = ut.loadingfile('hac', 'HACtfidfwithsvd_pipeline_cluster_labels.json')
printlabels(hac_tfidf_without_svd_prediction_file)

Cluster 0: Software Glitches and Stability Issues
Cluster 1: General Maintenance and Product Inquiries
Cluster 2: Account Recovery and Password Reset
Cluster 3: General Network Connection Failures
Cluster 4: Mechanical and Hardware Issues
Cluster 5: Application Crashes and Version Errors
Cluster 6: Firmware Updates and Shipping Inquiries
Cluster 7: Intermittent Internet and Router Stability
Cluster 8: UI Error Messages and Screen Flickering
Cluster 9: Data Security and Refund Concerns
Cluster 10: Accidental Deletion and Urgent Recovery
Cluster 11: Battery Performance and Power Issues
Cluster 12: File Retrieval and Document Loss
Cluster 13: Invalid Login Credentials and Access
Cluster 14: System Crash and Data Loss Recovery
Cluster 15: Home Wi-Fi Detection and Setup
Cluster 16: Installation Errors and Unexpected Loss
Cluster 17: Hardware Repair and Replacement Requests


### HAC with Embeddings

In [93]:
hac_embeddings_without_pca_prediction_file = ut.loadingfile('hac', 'HACembeddingswithoutpca_pipeline_cluster_labels.json')
printlabels(hac_embeddings_without_pca_prediction_file)

Cluster 0: Initial Network Setup and Sudden Battery Drain
Cluster 1: Post-Firmware Update Issues and Reset Attempts
Cluster 2: Strange Hardware Noises and Error Message Analysis
Cluster 3: Urgent Data Recovery and Accidental File Deletion
Cluster 4: Account Access Errors and Locked Profile Recovery
Cluster 5: Intermittent Internet Drops and ISP Troubleshooting
Cluster 6: General Service Inquiries and Data Security Safety
Cluster 7: Productivity Impacts and Battery Longevity Concerns
Cluster 8: User Interface Navigation and Support Site Feedback
Cluster 9: Critical Power Failures and Peculiar Screen Errors
Cluster 10: Official Charger Issues and Power Supply Problems
Cluster 11: Screen Flickering and Display Hardware Failures
Cluster 12: Software Bug Reporting and Data Loss Prevention
Cluster 13: Total Power Loss and System Response Failures
Cluster 14: Software Freezing and General Data Security Fears
Cluster 15: Forgotten Passwords and Reset Link Malfunctions
Cluster 16: Application C

### HAC with Embeddings PCA

In [94]:
hac_embeddings_with_pca_prediction_file = ut.loadingfile('hac', 'HACembeddingswithpca_cluster_labels.json')
printlabels(hac_embeddings_with_pca_prediction_file)

Cluster 0: Network Connectivity and General Data Security
Cluster 1: Urgent File Recovery and Hardware Repair Concerns
Cluster 2: Account Access Errors and Widespread Login Issues
Cluster 3: Inventory Inquiries and Post-Reset Troubleshooting
Cluster 4: Firmware Update Bugs and Version Compatibility
Cluster 5: Productivity Loss and Battery Life Degradation
Cluster 6: Intermittent Performance and Support Feedback
Cluster 7: Software Bug Reporting and Data Loss in Apps
Cluster 8: UI Navigation and E-commerce/Cart Assistance
Cluster 9: Critical Power Failure and Response Issues
Cluster 10: Official Charger Compatibility and Power Supply
Cluster 11: Display Hardware Issues and Screen Flickering
Cluster 12: Stable Internet Connection and ISP Connectivity
Cluster 13: Peculiar Error Messages and System Notifications
Cluster 14: Acoustic Anomalies and Physical Hardware Suspicions
Cluster 15: System Freezing and Security/Privacy Stability
Cluster 16: Forgotten Passwords and Reset Option Failures

## LDA

### LDA with TFIDF

In [95]:
ladtfidf_prediction_file = ut.loadingfile('lda', 'LDAtfidf_pipeline_cluster_labels.json')
printlabels(ladtfidf_prediction_file)

Cluster 0: Widespread Data Loss and Model Response Issues
Cluster 1: Community Forums and Online Troubleshooting Tutorials
Cluster 2: Data Security and Factory Reset Safety Concerns
Cluster 3: Network Connectivity and Productivity Impact
Cluster 4: Firmware Updates and Document/File Retrieval
Cluster 5: Home Wi-Fi Connectivity and Charger Troubleshooting
Cluster 6: Account Access and Login Credential Recovery
Cluster 7: Battery Life Degradation and Strange Hardware Noises
Cluster 8: UI Error Messages and Screen Pop-up Configurations
Cluster 9: Hardware Repair, Replacement, and Warranty Inquiries
Cluster 10: Intermittent Feature Performance and System Settings
Cluster 11: Application Bugs and Cache/Data Management
Cluster 12: Software Version Updates and Maintenance Status
Cluster 13: Product Manuals and Account Password Support


In [96]:
ladtfidf = ut.load_model('lda', 'lda_with_tfidf.joblib')
predictionsldatfidf = ladtfidf.transform(raw_data)
predictions_final_ldatfidf = np.argmax(predictionsldatfidf, axis=1)
predictionoflabels(predictions_final_ldatfidf, ladtfidf_prediction_file)

Cluster 6: Account Access and Login Credential Recovery
Cluster 12: Software Version Updates and Maintenance Status


Prediction

Cluster 6: Account Access and Login Credential Recovery  
Cluster 12: Software Version Updates and Maintenance Status

### LDA with countvectorizer

In [97]:
ldacountvectorizer_prediction_file = ut.loadingfile('lda', 'ladwithcountvectorizer_pipeline_cluster_labels.json')
printlabels(ldacountvectorizer_prediction_file)

Cluster 0: Widespread Data Loss and Model Performance
Cluster 1: Account Lockouts and Session Unlock Support
Cluster 2: Data Security, Privacy, and Cancellation
Cluster 3: Software Updates and Application Stability
Cluster 4: Productivity Hindrance and Battery Longevity
Cluster 5: Firmware Maintenance and Wi-Fi Connectivity
Cluster 6: Credential Validation and Access Recovery
Cluster 7: Hardware Noise and Website Support Resources
Cluster 8: UI Pop-ups and Screen Error Configurations
Cluster 9: Hardware Repair and Replacement Logistics
Cluster 10: Intermittent Application and Setting Behavior
Cluster 11: App Cache Management and Data Bugs
Cluster 12: Product Versioning and Charging Hardware
Cluster 13: Factory Reset and Account Password Recovery
Cluster 14: Community Tutorials and Peripheral Setup


In [98]:
ldacountvectorizer = ut.load_model('lda', 'lda_with_count_vectorizer.joblib')
ldacountvectorizer_prediction_file = ut.loadingfile('lda', 'ladwithcountvectorizer_pipeline_cluster_labels.json')
predictionsldacountvectorizer = ldacountvectorizer.transform(raw_data)
predictions_final_ldacountvectorizer = np.argmax(predictionsldacountvectorizer, axis=1)
predictionoflabels(predictions_final_ldacountvectorizer, ldacountvectorizer_prediction_file)

Cluster 1: Account Lockouts and Session Unlock Support
Cluster 3: Software Updates and Application Stability


Prediction  
Cluster 1: Account Lockouts and Session Unlock Support  
Cluster 3: Software Updates and Application Stability

### LDA with Gensim Topics

In [99]:
def predict_gensim_lda(model, dictionary, raw_docs):
    tokenized_data = [str(doc).split() for doc in raw_docs]
    bow_corpus = [dictionary.doc2bow(text) for text in tokenized_data]
    final_labels = []
    for bow in bow_corpus:
        topic_probs = model.get_document_topics(bow, minimum_probability=0.0)
        best_topic = max(topic_probs, key=lambda x: x[1])[0]
        final_labels.append(best_topic)
        
    return np.array(final_labels)

In [100]:
ldagensim_prediction_file = ut.loadingfile('lda', 'lda_gensim_topics_pipeline_cluster_labels.json')
printlabels(ldagensim_prediction_file)

Cluster 0: Software Updates and Payment Documentation
Cluster 1: System Performance and Network Installation
Cluster 2: Hardware Noise and Functionality Issues
Cluster 3: E-commerce Transactions and Shipping
Cluster 4: Login Credentials and UI Error Displays
Cluster 5: File Loss and Warranty Status Inquiries
Cluster 6: Data Security and Hardware Repair/Recovery
Cluster 7: Battery Degradation and Refund Requests
Cluster 8: Wi-Fi Connectivity and Troubleshooting Manuals
Cluster 9: Peripherals, Adapters, and Stock Availability
Cluster 10: Community Forums and Factory Reset Guides
Cluster 11: Firmware Releases and Purchase History
Cluster 12: App Cache Management and Software Bugs
Cluster 13: Application Stability and Bug Reporting
Cluster 14: Internet Stability and Router Troubleshooting
Cluster 15: Account Configuration and Login Lockouts
Cluster 16: Password Recovery and Urgent Data Deletion
Cluster 17: Display Flickering and Charging Hardware


In [101]:
ldagensim = ut.load_model('lda', 'lda_gensim_topics.joblib')
dictionarygensim = ut.load_model('lda', 'lda_gensim_topics_dict.joblib')
ldagensim_prediction_file = ut.loadingfile('lda', 'lda_gensim_topics_pipeline_cluster_labels.json')
predictions_final = predict_gensim_lda(ldagensim, dictionarygensim, raw_data)
predictionoflabels(predictions_final, ldagensim_prediction_file)

Cluster 16: Password Recovery and Urgent Data Deletion
Cluster 0: Software Updates and Payment Documentation


Prediction  
Cluster 16: Password Recovery and Urgent Data Deletion  
Cluster 0: Software Updates and Payment Documentation

## Bert Topic

In [102]:
def pipeline_string_cleaner(X):
    s = pd.Series(X).astype(str)
    cleaned = s[s.str.strip().str.lower().map(lambda d: d != "" and d != "nan")]
    return cleaned.tolist()

In [103]:
berttopic_prediction_file = ut.loadingfile('bert', 'bertopicpipeline_cluster_labels.json')
printlabels(berttopic_prediction_file)

Cluster -1: Outlier: General Product & Software Updates
Cluster 0: Unexpected Product Updates and Maintenance
Cluster 1: Network Connectivity and Troubleshooting
Cluster 2: Data Loss and File Recovery
Cluster 3: Account Access and Password Recovery
Cluster 4: Software Bug Reporting and Fixes
Cluster 5: System Error Messages and UI Pop-ups
Cluster 6: Data Security and Privacy Concerns
Cluster 7: Product Purchase and ID Logistics
Cluster 8: Power Cycle and Response Issues
Cluster 9: Hardware Display and Screen Flickering
Cluster 10: Software Glitches and System Freezing
Cluster 11: Charging and Battery Supply Issues
Cluster 12: Hardware Noise and Acoustic Anomalies


In [104]:
bertopic = ut.load_model('bert', 'bertopic_full_pipeline.joblib')
bertmodel = bertopic.named_steps['bertopic']
topic_labels, prob_bert = bertmodel.transform(raw_data)
predictionoflabels(topic_labels, berttopic_prediction_file)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cluster 3: Account Access and Password Recovery
Cluster 4: Software Bug Reporting and Fixes


Prediction  

Cluster 3: Account Access and Password Recovery  
Cluster 4: Software Bug Reporting and Fixes 